# 07. 공통 기준 테이블 만들기

전체 345,390행을 보존하고 제외 표시·파생컬럼·근거·임시 LABEL을 붙임. CP 단독 조건이나 LOT 전체 수준 차이로 1을 만들지 않음.
`processed/cal01_base_v1.csv` 하나가 공통 기준임. 분류·회귀 폴더에는 이 표를 읽는 정의만 둠.

In [ ]:
from pathlib import Path
import sys
import json
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'data/README.md').is_file() and (p / 'notebooks').is_dir())
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
from namelesspark_pipeline import load_sources, build_lot_context, build_maintenance_context, apply_user_exclusions
from namelesspark_weak_labels import build_weak_labels
from namelesspark_datasets import build_master, export_master, classification_inputs

PROCESSED = ROOT / 'data/processed'
REVIEW = PROCESSED / 'review/v1'
SIGNALS = PROCESSED / 'analysis/signals/v1'
FEATURE_FILE = PROCESSED / 'features/v1/06_features_1min.csv'
CANDIDATE_FILE = SIGNALS / '05_all_sensor_sustained_candidates.csv'
ROW_LINK_FILE = REVIEW / '05_domain_row_index.csv'
for path in [FEATURE_FILE, CANDIDATE_FILE, ROW_LINK_FILE]:
    if not path.is_file():
        raise FileNotFoundError(f'05·06을 먼저 실행해야 함: {path}')
sources = load_sources(ROOT)
marked, exclusions = apply_user_exclusions(sources['temp'])
lot_context = build_lot_context(sources['temp'], sources['event'])
marked['GRADE'] = marked.LOT_NO.map(lot_context.set_index('LOT_NO').GRADE)
maintenance, part_details = build_maintenance_context(sources['maint'], sources['partchg'], sources['part'])
features = pd.read_csv(FEATURE_FILE, encoding='utf-8-sig', parse_dates=['MEAS_DT'])
candidates = pd.read_csv(CANDIDATE_FILE, encoding='utf-8-sig', parse_dates=['START', 'END'])
row_links = pd.read_csv(ROW_LINK_FILE, encoding='utf-8-sig', parse_dates=['MEAS_DT'])

## 임시 0·1 정의

1은 강한 센서 지속 변화와 적격 정비 기록의 시간 연관이 함께 있는 행임. 0은 이번 규칙 미검출이며 확정 정상은 아님. 사용자 제외 행은 LABEL을 비워두고 학습에서 뺌.
공정 허용치·고장 원인을 확정한 정답이 아니며, 문턱과 시간 범위는 실험 가정임. 자세한 조건은 `metadata/v1/dataset.json`에 남김.

In [ ]:
row_labels, interval_decisions, label_policy = build_weak_labels(marked, candidates, sources, maintenance)
master, input_columns = build_master(sources, features, row_labels, row_links)
REVIEW.mkdir(parents=True, exist_ok=True)
source_files = [ROOT/'data'/name for name in [
    'T-CR1-CAL01_온도.csv', 'G-02_조업이벤트.csv', 'G-01_정기수리캘린더.csv',
    'G-01_설비마스터.csv', 'G-01_부품마스터.csv', 'G-02_정비이력.csv', 'G-02_부품교체이력.csv',
]] + [FEATURE_FILE, CANDIDATE_FILE, ROW_LINK_FILE,
      ROOT/'src/namelesspark_weak_labels.py', ROOT/'src/namelesspark_datasets.py',
      ROOT/'src/namelesspark_pipeline.py', ROOT/'src/namelesspark_temporal_review.py']
master_path, manifest = export_master(ROOT, master, input_columns, label_policy, source_files)
interval_decisions.to_csv(REVIEW / 'weak_label_interval_decisions.csv', index=False, encoding='utf-8-sig')
print('공통 기준:', master_path)
print('전체:', len(master), '/ 임시 0:', manifest['label_0'], '/ 임시 1:', manifest['label_1'])
print('사용자 제외:', manifest['excluded_rows'], '/ 입력 완전한 학습 후보:', manifest['model_eligible_rows'])
print('임시 1의 센서별 기여:', label_policy['positive_contribution_by_sensor'])

## 컬럼은 이렇게 보면 됨

전체 106개 중 RF 입력 후보 67개, 목표값 LABEL 1개, 식별·검토용 38개임. CSV 전체를 X에 넣는 것은 아님.

| 컬럼·묶음 | 의미와 사용 |
| --- | --- |
| ROW_ID / LOT_NO / MEAS_DT / GRADE | 행·LOT·시각·강종 식별. 분할·결과 확인용 |
| CLN-IN / TC-* / OP-* / CP-Z4 | 세정 입측 온도·각 존 온도·제어출력·CP 기록값. 원본 21계측 입력 후보 |
| CP-Z4M | 해석·생성 관계 확인이 남은 보조계측. 원본만 보존하고 X에서 제외 |
| LOT_ELAPSED_MIN / FEATURE_MINUTE | 원본 LOT 첫 관측 후 경과 분 / 과거 특징을 붙일 기준 분. 전자는 기존 67개 입력 후보에 포함, 후자는 조인용 |
| GRADE_* | 해당 강종이면 1, 아니면 0인 7개 컬럼. 정상·이상 표시가 아님 |
| 센서_1분전 / _5분전 | 1분 전 / 5분 전의 분 평균 |
| 센서_직전10분평균 / _직전10분표준편차 | 직전 10개 분 평균의 평균 / 표준편차. 현재 분은 제외 |
| 센서_직전두분차이 | 직전 분 평균 − 2분 전 평균. 시간으로 나눈 변화율은 아님 |
| OP-Z3/Z4/Z5_직전10분포화율 | 직전 10분의 유효 원본 관측 중 출력 100 이상 비율(%) |
| EXCLUDED / EXCLUSION_REASON | 사용자 지정 제외 여부 / 이유. 제외 행의 LABEL은 비움 |
| FEATURES_COMPLETE / MODEL_ELIGIBLE | 입력 67개가 모두 유한한 숫자인지 / 제외되지 않고 입력과 LABEL이 있는지. 학습 완료 표시는 아님 |
| LABEL / LABEL_STATUS / LABEL_VERSION | 임시 0·1 / 판정 상태 / 규칙 버전. LABEL만 y이며 X에는 넣지 않음 |
| LABEL_REASON / RULE / EVIDENCE_CONFIDENCE | 판정 이유·규칙·근거 한계 설명. 확률 점수가 아니며 X 제외 |
| EVIDENCE_WO / MNT_TYPE / SENSOR / CANDIDATE_IDS | 실제 컬럼은 모두 EVIDENCE_ 접두어임. 1 판정에 사용한 정비번호·정비유형·센서·후보 ID |
| EVIDENCE_DOWN_MIN / HAS_PART_RECORD / PART_NAMES / PART_QTY_TOTAL / PART_CHANGE_REASONS / FAIL_MODE | 모두 EVIDENCE_ 접두어임. 근거 정비의 정지 분량·교체기록 유무·부품명·수량·사유·고장모드. WO:값 형식이며 X 제외 |
| ASSOCIATION_KIND / EVIDENCE_INPUT_ALLOWED | 감지 전24시간·작업 종료 후24시간의 연결 유형 / 근거의 X 사용 금지(False) 표시 |
| ANY_SENSOR_CANDIDATE / 각 *_CONTEXT / MAINT_WORK_OVERLAP / GRADE_CHANGE_NEARBY_30MIN | 후보·제어반응·주기·LOT 경계·수리·작업·강종변경 맥락. 모두 검토용이며 이상 확정이나 실제 정지 표시가 아님 |
| LOT_CONTEXT_ID / SENSOR_CANDIDATE_IDS | LOT의 사건 근거 / 겹치는 전체 센서 후보를 찾아가는 ID. 채택 후보만 담은 EVIDENCE_CANDIDATE_IDS와 구분 |

과거 5종은 TC/OP-Z3·Z4·Z5와 CP-Z4의 7개 센서에 붙어 35개, OP 포화율 3개를 더해 총 38개임. 근거 문자열은 인코딩해도 X에서 제외함. DOWN_MIN은 정지 분량이지 해당 행 시각의 정지 여부가 아님.


## 모델 입력 구분

CSV 전체를 그대로 모델에 넣으면 안 됨. `input_columns`는 기존 RF 실험을 위한 수치 입력 후보 67개이며, 모든 방법에 적합하다고 검증된 목록은 아님.

| 방법 | 현재 상태 |
| --- | --- |
| RandomForestClassifier | 센서21 + 과거 특징38 + LOT 경과1 + 강종 원핫7의 67개 후보. 학습·성능 검증 전 |
| Z-score | 점수를 계산할 센서·파생컬럼과 강종별 비교 집단을 아직 정하지 않았음 |
| IsolationForest | 입력 선택·학습 조건을 아직 정하지 않았음 |
| 회귀 | 목표 센서·예측 시차 미정 |

텍스트 근거·ID·LABEL은 점수 입력이 아님. 근거는 인코딩해도 입력에서 제외함. 강종 원핫과 LOT 경과시간도 수치라는 이유로 Z-score에 그대로 넣지 않음. Z-score는 학습 구간의 평균·표준편차, 상수 컬럼 처리, 임계값을 먼저 정해야 함.
같은 기준표와 비교 행을 공유할 수 있지만 방법별 입력 검토는 남아 있음. 분할·9:1 추출·학습·평가는 아직 안 했음.

In [ ]:
X, y, row_keys = classification_inputs(master, input_columns)
assert X.select_dtypes(include='number').shape[1] == X.shape[1]
assert X.notna().all().all() and set(y.unique()).issubset({0, 1})
assert X.shape[1] == 67
assert not set(['LABEL', 'RULE', 'EVIDENCE_WO', 'MEAS_DT', 'LOT_NO', 'EXCLUDED']).intersection(X.columns)

# 수치 후보표 준비와 각 알고리즘에 대한 입력 검증은 구분함.
contract_path = PROCESSED / 'classification/v1/dataset.json'
contract = json.loads(contract_path.read_text(encoding='utf-8'))
contract['method_input_columns'] = {
    'RandomForestClassifier': input_columns,
    'IsolationForest': None,
    'Z-score': None,
}
contract['method_input_status'] = {
    'RandomForestClassifier': 'candidate_columns_only_not_trained_or_validated',
    'IsolationForest': 'input_selection_pending',
    'Z-score': 'input_selection_grouping_and_threshold_pending',
}
contract['evaluation'] = '같은 기준표와 평가 행을 공유할 수 있으나 방법별 입력 검토가 남아 있음. LABEL은 X와 Z-score·IF fit에서 제외함.'
contract_path.write_text(json.dumps(contract, ensure_ascii=False, indent=2), encoding='utf-8')
print('RF 수치 입력 후보:', X.shape, '/ y:', y.value_counts().to_dict())
print('Z-score·IF 입력 선택과 모든 모델 학습·평가는 아직 안 했음')
print('분류 정의:', contract_path)
print('회귀 정의:', PROCESSED / 'regression/v1/dataset.json')